# GEOG 412: Programming for Geospatial Data Science, Unit 4

## Assignment 4: Spatial Analysis with Python

For this assignment, you will...

*   Complete a series of five data analysis exercises designed to help you practice basic spatial analysis and mapping techniques with *GeoPandas* in a targeted manner (ten points altogether);
*   Complete one more complicated exercise that requires you to creatively apply what you have learned about the logic and syntax of Python programming in a less structured manner (ten points).



Your code should be written directly within a copy of this notebook, and you should upload a copy of your notebook to the course website as you have done for previous assignments.

**To begin, choose "Save a copy in Drive" from the File menu to create your own copy of this notebook.**

Let's begin by installing some dependencies (run every time your Python runtime clears)...

In [ ]:
!pip install pandas geopandas geoplot mapclassify contextily

## Spatial Analysis and Mapping Exercises

Each one of these exercises will involve practicing specific elements of the spatial analysis and visualization techniques that are covered in this unit.  These exercises are designed to ensure that you are familiar with common methods associated with the use of basemaps as well as various features of GeoPandas.

**Please note that these exercises require the use of various assignment and demonstration data files.**  These data files are all available on the course website and must be uploaded to your Colab environment.

###Exercise 1 (one point)

Generate a simple map of Yosemite National Park with a suitable terrain basemap.  Show the park's boundaries as a hollow polygon.

In [ ]:
# Code for exercise 1 goes here

import geopandas as gpd
import contextily as cx
import matplotlib.pyplot as plt
import matplotlib.colors as cl
import mapclassify

parks = gpd.read_file('national_parks.gpkg')

yosemite = parks.query("unit_code =='YOSE'")

fig, ax = plt.subplots(figsize=(12,12))
ax.set_title('Yosemite National Park',fontsize=24)
ax.set_axis_off()

yosemite_map = yosemite.plot(
    ax=ax,
    figsize=(12,12),
    facecolor='none',
    edgecolor='green',
    linewidth=2)
cx.add_basemap(yosemite_map,crs=yosemite.crs.to_string(),source=cx.providers.Esri.WorldTopoMap)

###Exercise 2  (one point)

Use GeoPandas to perform a point-in-polygon spatial join connecting Airbnb rentals to their containing Los Angeles County cities and communities.

In [ ]:
# Code for exercise 2 goes here

!sudo apt install -y libspatialindex-dev
!pip install rtree

import pandas as pd
import geopandas as gpd
import contextily as cx

listings = pd.read_csv('listings.csv')
neighbourhoods = gpd.read_file('neighbourhoods.geojson')

listings = gpd.GeoDataFrame(
    listings,
    geometry=gpd.points_from_xy(listings.longitude, listings.latitude),
    crs=4326) # pd df to gpd gdf

listings = listings.drop(columns=['neighbourhood','neighbourhood_group'])

nhoods_joined = neighbourhoods.sjoin(listings,how='left')

nhoods_joined["count"] = 1

nhoods_joined = nhoods_joined.groupby('neighbourhood').agg({'geometry':'first','count':'sum'})

nhoods_joined.set_geometry("geometry")

nhoods_joined.head()

###Exercise 3  (two points)

Use GeoPandas to find the average nightly rate of Airbnb rentals within Los Angeles County cities and communities.  Sort the list of communities in descending order.  Create a choropleth map of these data as well, depicting average nightly rate by community.

*Hint:  You will need to use a spatial join.*

In [ ]:
# Code for exercise 3 goes here

import pandas as pd
import geopandas as gpd
import contextily as cx
import matplotlib.pyplot as plt
import matplotlib.colors as cl
import mapclassify

listings = pd.read_csv('listings.csv')
neighbourhoods = gpd.read_file('neighbourhoods.geojson')

listings = gpd.GeoDataFrame(
    listings,
    geometry=gpd.points_from_xy(listings.longitude, listings.latitude),
    crs=4326) # pd df to gpd gdf

listings = listings.drop(columns=['neighbourhood','neighbourhood_group'])

nhoods_joined = neighbourhoods.sjoin(listings,how='left')

nhoods_joined["count"] = 1

nhoods_joined = nhoods_joined.groupby('neighbourhood').agg({'geometry':'first','price':'mean'})
nhoods_joined = nhoods_joined.set_geometry("geometry")
nhoods_joined = nhoods_joined.set_crs(epsg=4326)

fig, ax = plt.subplots(figsize=(12,12))
ax.set_title('LA Communities by Avg AirBnB Rate',fontsize=24)
ax.set_axis_off()

nhoods_map = nhoods_joined.plot(
    ax=ax,
    column="price",
    legend=True,
    figsize=(12,12),
    cmap="Reds",
    scheme="NaturalBreaks",
    k=5,
    edgecolor='black',
    linewidth=0.5)
cx.add_basemap(nhoods_map,crs=nhoods_joined.crs.to_string(),source=cx.providers.Esri.WorldStreetMap)

nhoods_joined.sort_values(by="price",ascending=False)

###Exercise 4 (two points)

Use GeoPandas to create a polygon (multi-part polygon permissible) feature showing all areas within Los Angeles County that are within 1,000 feet of an active Airbnb.  Your script should generate this polygon and then save it as a GeoPackage file.

In [ ]:
# Code for exercise 4 goes here

import pandas as pd
import geopandas as gpd
import contextily as cx
import fiona

listings = pd.read_csv('listings.csv')
listings = gpd.GeoDataFrame(listings,geometry=gpd.points_from_xy(listings.longitude,listings.latitude),crs=4326)

listings = listings.to_crs(3310)

buffer = listings.buffer(304.8)

buffer_df = gpd.GeoDataFrame(buffer,geometry=buffer)
buffer_df_dissolved = buffer_df.dissolve()

buffer_df_dissolved['geometry'].to_file("la_county_airbnb_buffer.gpkg", driver="GPKG", layer="geometry")

###Exercise 5  (four points)

Find a point data source and a polygon data source, both of which should be (1) thematically related, (2) readable by GeoPandas, and (3) available from static URLs.  

In your script for this exercise, you should do the following (at least):

1.   Read the data directly from their source URLs.
2.   Generate a map of the point data with a suitable basemap.
3.   Generate either a kernel density map OR a quadtree map of the point data.
4.   Perform a point-in-polygon join and create a map that displays some outcome of the spatial join.

In [ ]:
# reset runtime and install deprecated shapely for easy kernel density plot

!pip install shapely==1.8.5.post1 pandas geopandas geoplot mapclassify contextily
!sudo apt install -y libspatialindex-dev
!pip install rtree

In [ ]:
# Code for exercise 5 goes here

import pandas as pd
import geopandas as gpd
import contextily as cx
import matplotlib.pyplot as plt
import matplotlib.colors as cl
import mapclassify
import geoplot as gplt
import rtree

!wget = https://www2.census.gov/geo/tiger/TIGER2020/POINTLM/tl_2020_06_pointlm.zip
!wget = https://www2.census.gov/geo/tiger/TIGER2020/TRACT/tl_2020_06_tract.zip

!unzip tl_2020_06_pointlm.zip
!unzip tl_2020_06_tract.zip

landmarks = gpd.read_file("tl_2020_06_pointlm.shp")
tracts = gpd.read_file("tl_2020_06_tract.shp")

# point data map

fig, ax = plt.subplots(figsize=(12,12))
ax.set_title('Landmarks in California',fontsize=24)
ax.set_axis_off()
landmark_map1 = landmarks.plot(ax=ax,figsize=(12,12),markersize=0.1)
cx.add_basemap(landmark_map1,crs=landmarks.crs,source=cx.providers.Esri.WorldStreetMap)

# kernel density map

landmarks_kd = gplt.kdeplot(landmarks,figsize=(12,12),shade=True,clip=tracts.dissolve())
gplt.polyplot(tracts, zorder=1, ax=landmarks_kd)

# spatial join

tracts_joined = tracts.sjoin(landmarks,how='left')

tracts_joined["count"] = 1
tracts_joined = tracts_joined.groupby('GEOID').agg({'geometry':'first','count':'sum'})
tracts_joined = tracts_joined.set_geometry("geometry")
tracts_joined = tracts_joined.set_crs(epsg=4326)


fig, ax = plt.subplots(figsize=(12,12))
ax.set_title('California Tracts by Landmark Count',fontsize=24)
ax.set_axis_off()
tracts_map = tracts_joined.plot(
    ax=ax,column="count",
    legend=True,
    figsize=(12,12),
    cmap="Reds",
    edgecolor='black',
    linewidth=0.5)
cx.add_basemap(tracts_map,crs=tracts_joined.crs,source=cx.providers.Esri.WorldStreetMap)

##Python Programming Challenge

This challenging exercise is designed to put your new Python programming skills to the test.  As you complete this exercise you will need to carefully think about not only the specific techniques that you will need to use to write code that meets the requirements of the challenges, but also the logical sequences and control flow that are required in order for your script to function properly.  Don't be afraid to get creative, and make sure to test your script thoroughly before submitting.

###Vacation Rental Analysis Tool

For this week's assignment, you will develop an application that is designed to help users explore the characteristics of Airbnb vacation rentals in a region of your choice.  A **region** for the purposes of this assignment is one of the regional datasets avaiable for download from [Inside Airbnb](http://insideairbnb.com/get-the-data.html).  Examples of regions available for download are: Los Angeles, California and Clark County, Nevada.

Your application should contain two different components:  a region-wide data exploration tool, and a local area analytics tool.  For the sake of simplicity, it is advisable that you write these two components of your application separately.

####Component 1: Region-Wide Data Exploration Tool

This component will generate five region-wide visualizations of your data.  This component of your program must include the following elements:

*   A map of all point locations of Airbnb rentals throughout the region, configured with a suitable and unobtrusive basemap.
*   A kernel density map showing distribution of Airbnb rentals throughout the region.
*   A map showing the average nightly rate of Airbnb rentals within each of the region's neighborhoods.
*   A histogram (with up to 20 bins) showing the distribution of nightly rates of Airbnb rentals region-wide.

This component of your script should essentially generate a single report of these five visualizations, and no user interface elements are required.

####Component 2: Local Area Analytics Tool

This component will generate some useful information and visualizations to help users understand the vacation rentals within the vicinity of a given address.  The following components are required for this tool:

* A map showing the locations of Airbnb rentals within the specified radius of the user's chosen location.
* A text caption indicating (1) the number of Airbnb rentals within the specified radius, (2) the average nightly rate of rentals within the radius, and (3) how that average rate compares to the regional average.
* Two other elements (visualizations, dynamic text, etc.) of your choice -- see this unit's geospatial forum for more information.

A simple user interface will be required for this application.  Included in the user interface should be:

* A text field wherein the user will enter an address of focus, which will be geocoded.
* A text field wherein the user will enter a radius (in meters) for the local area analysis.
* A button to trigger the execution of the tool.

When clicking the button to execute the tool, all data should be redrawn as necessary, clearing the results of any previous operations.

**Component 1: Region-Wide Data Exploration Tool**

In [ ]:
# probably not necessary if just executed for 'exercise 5' but here nonetheless:
#
# reset runtime and install deprecated shapely for easy kernel density plot


!pip install shapely==1.8.5.post1 pandas geopandas geoplot mapclassify contextily
!sudo apt install -y libspatialindex-dev
!pip install rtree

In [ ]:
# Code for county-wide data exploration component of challenge exercise goes here

import pandas as pd
import geopandas as gpd
import contextily as cx
import matplotlib.pyplot as plt
import matplotlib.colors as cl
import mapclassify
import rtree
import geoplot as gplt
from geopy.geocoders import Nominatim

# read files

ams_listings = pd.read_csv('ams_listings.csv')
ams_neighbourhoods = gpd.read_file('ams_neighbourhoods.geojson')
ams_listings = gpd.GeoDataFrame(ams_listings,geometry=gpd.points_from_xy(ams_listings.longitude, ams_listings.latitude),crs=4326) # pd df to gpd gdf

# vis 1

ams_listings_reproj = ams_listings.to_crs(crs="28992")

fig, ax = plt.subplots(figsize=(12,12))
ax.set_title("AirBnB's in Amsterdam",fontsize=24)
ax.set_axis_off()
ams_listings_map = ams_listings_reproj.plot(
  ax=ax,
  figsize=(12,12),
  markersize=0.1)
cx.add_basemap(ams_listings_map,crs=ams_listings_reproj.crs,source=cx.providers.Esri.WorldStreetMap)

# vis 2

ams_listings = ams_listings.drop(columns=['neighbourhood','neighbourhood_group'])

ams_nhoods_joined = ams_neighbourhoods.sjoin(ams_listings,how='left')
ams_nhoods_joined["count"] = 1
ams_nhoods_joined = ams_nhoods_joined.groupby('neighbourhood').agg({'geometry':'first','price':'mean'})

ams_nhoods_joined = ams_nhoods_joined.set_geometry("geometry")
ams_nhoods_joined = ams_nhoods_joined.set_crs(epsg=4326)
ams_nhoods_reproj = ams_nhoods_joined.to_crs(crs="28992")

fig, ax = plt.subplots(figsize=(12,12))
ax.set_title('Amsterdam Neighbourhoods by Avg AirBnB Rate',fontsize=24)
ax.set_axis_off()
ams_nhoods_map = ams_nhoods_reproj.plot(
    ax=ax,
    column="price",
    legend=True,
    figsize=(12,12),
    cmap="Reds",
    scheme="NaturalBreaks",
    k=5,
    edgecolor='black',
    linewidth=0.5)
cx.add_basemap(ams_nhoods_map,crs=ams_nhoods_reproj.crs.to_string(),source=cx.providers.Esri.WorldStreetMap)

# vis 3

ams_listings_kd = gplt.kdeplot(ams_listings,figsize=(12,12),shade=True,clip=ams_neighbourhoods.dissolve())
gplt.polyplot(ams_neighbourhoods, zorder=1, ax=ams_listings_kd)

# vis 4

ams_nhoods_joined.hist(
      column='price',
      figsize=(12,6))

**Component 2: Local Area Analytics Tool**

In [ ]:
# if not installed yet somehow

!pip install geopy pandas geopandas geoplot mapclassify contextily
!sudo apt install -y libspatialindex-dev
!pip install rtree

**Input geocoded places must be in Amsterdam, The Netherlands**
*some example searches*
*   Bunk Hotel Amsterdam
*   Van Gogh Museum Amsterdam
*   Amsterdam Centraal Station
*   Anne Frank House Amsterdam
*   Vondelpark Amsterdam

**FOR THIS SCRIPT TO WORK, MAKE SURE THERE IS NO EXISTING 'listing.csv' FILE IN MEMORY.**

In [ ]:
# Code for local area analytics component of challenge exercise goes here

import pandas as pd
import geopandas as gpd
import contextily as cx
import numpy as np
import math
import mapclassify
import matplotlib.pyplot as plt
import matplotlib.colors as cl
import rtree
import ipywidgets as widgets
from IPython.display import display, clear_output
from geopy.geocoders import Nominatim
from shapely.geometry import shape, Point

# files to gdf
!wget = http://data.insideairbnb.com/the-netherlands/north-holland/amsterdam/2023-12-12/visualisations/listings.csv
ams_listings = pd.read_csv("listings.csv")
ams_listings = gpd.GeoDataFrame(ams_listings,geometry=gpd.points_from_xy(ams_listings.longitude, ams_listings.latitude),crs=4326) # pd df to gpd gdf
ams_listings = ams_listings.set_geometry("geometry",crs="EPSG:4326") # set geo
ams_listings = ams_listings.set_crs(epsg=4326) # set crs
ams_listings.crs = "EPSG:4326" # direct crs
ams_listings = gpd.GeoDataFrame(ams_listings,geometry=ams_listings.geometry,crs=4326) # confirm

# geocoder
geocoder = Nominatim(user_agent='local_area_analysis')

# dropdown selectors
address_field = widgets.Text(value=input("Address: "),placeholder="Address: ",description="Address: ",disabled=False)
buffer_field = widgets.FloatText(value=float(input("Radius in meters (500m - 10,000m): ")),description="Radius <m>: ",disabled=False)
go_button = widgets.ToggleButton(value=False,description="Go!",disabled=False,button_style="",tooltip="Description",icon="")

# grid
grid = widgets.GridspecLayout(1,3,height="60px")
grid[0,0] = address_field
grid[0,1] = buffer_field
grid[0,2] = go_button

# on change event / function
def on_change(event=None):
  clear_output()

  if buffer_field.value < 500:
    print("Please enter a radius of more than 500 meters.")
    display(grid)
  elif buffer_field.value > 10000:
    print("Please enter a radius of less than 10 kilometers.")
    display(grid)
  else:
    print("Selected place: ",address_field.value)
    print("Defined radius: ",buffer_field.value)

    user_place = geocoder.geocode(address_field.value)
    user_pt = {"address":[user_place.address], "geometry":[Point(user_place.longitude, user_place.latitude)]}
    user_gdf = gpd.GeoDataFrame(user_pt, crs=4326)

    user_gdf_3310 = user_gdf.to_crs(3310) # reproject for buffering
    buffer = user_gdf_3310.buffer(buffer_field.value)
    buffer_df = gpd.GeoDataFrame(buffer,geometry=buffer)
    buffer_df_dissolved = buffer_df.dissolve()

    buffer_4326 = buffer_df_dissolved.to_crs(crs="4326") # back to 4326 for sjoin
    buffer_4326 = buffer_4326.set_geometry("geometry",crs="EPSG:4326") # set geo
    buffer_4326 = buffer_4326.set_crs(epsg=4326) # set crs
    buffer_4326.crs = "EPSG:4326" # direct crs
    buffer_4326 = gpd.GeoDataFrame(buffer_4326, geometry=buffer_4326.geometry, crs=4326) # confirm

    buffer_4326_join = gpd.sjoin(left_df=ams_listings,right_df=buffer_4326,how="inner",predicate="intersects")
    buffer_4326_join = buffer_4326_join.set_geometry("geometry",crs="EPSG:4326") # set geo
    buffer_4326_join = buffer_4326_join.set_crs(epsg=4326) # set crs
    buffer_4326_join.crs = "EPSG:4326" # direct crs
    buffer_4326_join = gpd.GeoDataFrame(buffer_4326_join, geometry=buffer_4326_join.geometry, crs=4326) # confirm

    buffer_to_map = buffer_4326_join.to_crs(crs="28992") # project to local dutch
    buffer_to_map = buffer_to_map.set_geometry("geometry",crs="EPSG:28992") # set geo
    buffer_to_map = buffer_to_map.set_crs(epsg=28992) # set crs
    buffer_to_map.crs = "EPSG:28992" # direct crs
    buffer_to_map = gpd.GeoDataFrame(buffer_to_map, geometry=buffer_to_map.geometry, crs=28992) # confirm

    fig, ax = plt.subplots(figsize=(16,6))
    ax.set_title("AirBnbs within " + str(buffer_field.value) + " meters of " + str(address_field.value),fontsize=16)
    ax.set_axis_off()
    local_map = buffer_to_map.plot(ax=ax,color="Red",figsize=(16,16),markersize=0.3) # plot
    cx.add_basemap(local_map,crs=buffer_to_map.crs,source=cx.providers.Esri.WorldStreetMap)

    print("There are " + str(len(buffer_to_map.index)) + " AirBnB rentals active within " + str(buffer_field.value) + " meters of " + str(address_field.value) + ".") # count

    local_avg_price = buffer_to_map["price"].mean(axis=0) # price
    local_avg_price = math.trunc(local_avg_price)
    region_avg_price = ams_listings["price"].mean(axis=0)
    region_avg_price = math.trunc(region_avg_price)
    print("These " + str(len(buffer_to_map.index)) + " rentals cost " + str(local_avg_price) + " USD per night, on average.")
    print("Compare this to the average nightly rate for all of Amsterdam: " + str(region_avg_price) + " USD per night.")

    local_rpm = buffer_to_map["reviews_per_month"].sum(axis=0) # activity
    local_rpm = math.trunc(local_rpm)
    region_rpm = ams_listings["reviews_per_month"].sum(axis=0)
    region_rpm = math.trunc(region_rpm)
    pct_city_activity = ( local_rpm / region_rpm ) * 100
    pct_city_activity = math.trunc(pct_city_activity)
    print("Together these rentals receive up to " + str(local_rpm) + " reviews per month, on average.")
    print("That amounts to " + str(pct_city_activity) + "% of the city's monthly user-review activity on AirBnB.")
    print("Edit your search parameters and click Go! to view a new region.")

    display(grid)

# observe
go_button.observe(on_change,names="value")

# start
display(grid)